# Chapter 23 — Frontier Safety 정책 — 능력 임계와 안전장치 게이트

> **Chapter 23 · Part III**

능력 점수를 임계 단계에 매핑해 요구 안전장치를 결정하는 **게이트 패턴**을 익히고, 경계값 전이·정책이 도입 속도에 주는 영향·불확실 구간의 예방원칙을 코드로 확인합니다.

## 실습 목표
1. 능력 점수를 임계 단계에 매핑하고, **경계값(0.5, 0.8)과 그 근처**에서 등급이 어떻게 전이하는지 확인합니다.
2. 단계별 필수 안전장치를 산출합니다.
3. 벤더가 모델을 상위 등급으로 재분류하면 사내 도입 일정이 밀리는 것을 시뮬레이션하고, 불확실 구간이 임계에 걸치면 상위 등급으로 보는 예방원칙을 적용합니다.

> **주의 — 이 표는 가상 서사입니다.** 아래 `THRESHOLDS` 의 요구사항은 이 책이
> ch08 에서 그린 가상 시나리오(Project Glasswing) 용 예시일 뿐, 현행 Anthropic RSP 의 인용이
> 아닙니다. 본문 §23.2.2 가 밝히듯 **RSP v3.x 는 계단식 ASL 요구사항 표를 두지 않습니다** —
> 사내 문서에 'ASL-4 이면 이렇게 한다' 를 적는 것은 현재 근거가 없습니다. 여기서 배우는 것은
> 특정 등급표가 아니라 '능력이 임계를 넘으면 안전장치가 발동한다' 는 **게이트 구조**입니다.
> 그래서 등급 이름도 벤더 라벨이 아니라 중립 라벨(L0~L2)을 씁니다. 데모는 표준 라이브러리만으로 동작합니다.

In [ ]:
# 표준 라이브러리만으로 동작 — 설치 불필요

In [ ]:
# (가상) 능력 임계 → 안전장치 매핑. 현행 RSP 인용이 아니라 게이트 패턴 예시.
# 본문 §23.2.2: RSP v3.x 에는 이런 계단식 ASL 요구사항 표가 없다.
# 각 등급에는 그 안전장치를 갖추는 데 필요한 사내 도입 준비 기간(lead_days)도 붙였다.
THRESHOLDS = [
  (0.0, 'L0 (기본)',   ['기본 사용정책'],                         0),
  (0.5, 'L1 (제한)',   ['접근 제한', '감사 로그', '탈옥 상시검증'], 30),
  (0.8, 'L2 (고위험)', ['제한 배포', '수출통제 검토', '전담 안전팀'], 90),
]

---
## Step 1. 능력 임계 게이트 — 경계값과 전이

능력이 높을수록 더 강한 안전장치를 요구한다는 **구조**를 코드로 표현합니다. 등급 이름(L0~L2)은
특정 벤더 정책이 아니라 이 게이트를 설명하기 위한 중립 라벨입니다. 게이트의 실제 동작은
경계값(0.5, 0.8)에서 갈리므로, 경계 바로 아래·위(0.49/0.51, 0.79/0.81)를 넣어 등급 전이를
직접 확인합니다.

In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: 기본값은 가장 낮은 등급(THRESHOLDS 의 첫 항목) — 특정 벤더 라벨을 기본값으로 두지 않는다

# 경계값(0.5, 0.8)과 그 근처에서 등급이 전이하는 것을 확인
print(f"{'능력점수':>8}  {'등급':>10}  요구 안전장치")
print('-' * 60)
for score in (0.0, 0.49, 0.50, 0.51, 0.79, 0.80, 0.81, 0.95):
    lv, sg, _ = safety_gate(score)
    print(f"{score:8.2f}  {lv:>10}  {sg}")

<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
def safety_gate(capability_score):
    # 기본값은 가장 낮은 등급(THRESHOLDS 의 첫 항목) — 특정 벤더 라벨을 기본값으로 두지 않는다
    thr0, lv0, sg0, days0 = ...
    level, safeguards, lead_days = ...
    for thr, lv, sg, days in THRESHOLDS:
        if capability_score >= thr:
            level, safeguards, lead_days = ...
    return ...
```
</details>


---
## Step 2. 정책이 도입 속도를 결정한다 — 재분류 시뮬레이션

벤더가 같은 모델의 능력 추정치를 올려 **상위 등급으로 재분류**하면, 사내에서 요구되는
안전장치가 늘고 그만큼 도입 일정이 밀립니다. 등급별 준비 기간(`lead_days`)으로 그 지연을
계산해 봅니다.

In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: def adoption_plan(old_score, new_score):

# 벤더가 능력 추정치를 0.45 → 0.82 로 올려 재분류한 경우
old_lv, new_lv, new_sg, lead, slip = adoption_plan(0.45, 0.82)
print(f'재분류: {old_lv} → {new_lv}')
print(f'새 등급 요구 안전장치: {new_sg}')
print(f'도입 준비 기간: {lead}일 (이전 대비 +{slip}일 지연)')
print('→ 벤더 정책(등급) 변화가 사내 도입 속도를 그대로 결정합니다.')

<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
def adoption_plan(old_score, new_score):
    old_lv, _, old_days = ...
    new_lv, new_sg, new_days = ...
    slip = ...
    return ...
```
</details>


---
## Step 3. 불확실 구간의 예방원칙

능력 평가에는 오차가 있습니다. 추정치에 불확실 구간(± 오차)이 있고 그 구간이 임계값을
**걸치면**, 낮은 등급으로 확정하기 전에 **상위 등급으로 보수적으로 취급**하는 것이 예방원칙입니다.
점 추정만 보면 놓칠 위험을, 구간을 봄으로써 잡습니다.

In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: def precautionary_gate(estimate, margin):

# 추정치 0.78 ± 0.05 → 구간 [0.73, 0.83] 이 0.8 임계를 걸친다
point_lv, upper_lv, upper_sg, escalated = precautionary_gate(0.78, 0.05)
print(f'점 추정(0.78) 등급: {point_lv}')
print(f'구간 상한(0.83) 등급: {upper_lv}  (예방원칙 상향 적용: {escalated})')
print(f'적용 안전장치: {upper_sg}')
print('→ 임계에 걸친 불확실 구간은 상위 등급으로 취급해 과소 대응을 막습니다.')

<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
def precautionary_gate(estimate, margin):
    ...
    point_lv, _, _ = ...
    upper_lv, upper_sg, _ = ...
    escalated = ...
    return ...
```
</details>


---
## 정리
- 능력이 임계를 넘으면 접근 제한·수출통제(ch08 §8.1.4)·전담팀 같은 안전장치가 발동합니다.
  게이트의 동작은 경계값(0.5, 0.8)에서 갈리므로, 경계 근처를 반드시 테스트해야 합니다.
- 핵심은 특정 등급표가 아니라 **'능력→안전장치' 게이트 구조**입니다. 그래서 기본값도 중립
  라벨(L0)이며, 실제 벤더 정책(RSP·PF·FSF)의 최신 요구사항은 본문 §23.2~§23.4 와 각 벤더 원문을 확인하세요.
- 벤더가 모델을 상위 등급으로 재분류하면 요구 안전장치가 늘어 **사내 도입 일정이 그만큼 밀립니다** — 정책 모니터링을 정기 운영 업무로 둡니다.
- 능력 평가의 불확실 구간이 임계에 걸치면 **상위 등급으로 보수적으로 취급**(예방원칙)해 과소 대응을 막습니다.

> **다음 Part:** 미토스/페이블5 lab.